# I'll be breaking the code into blocks by assignment
This first assignment is gonna be a big block, essentially what I had to do 

In [4]:
import requests
import pandas as pd

url = "http://ws.audioscrobbler.com/2.0/" #Url base for the API, used in my future F strings as inputs, F strings are awesome
YOUR_API_KEY = '915b23d080ae463f6850ac8cec3aefb6' #my API key,
artist='Cher'
#_________________________________________________________________________________________________________________________________________
#BLOCK 1 - Prime Data, find how many weeks and how to grab tags
#This below block gives me both the tag grabber for each of the artists I'm looking at as well as the weeks I'm counting. This is important
#because I'm looking to do a for loop in a for loop, essentially the main loop is going to cycle weeks and grab the listening data for that 
#week, the second loop is going to cycle between either the top 250 artists or the length of the amount of artists, whichever is smaller
#in order to get the weeks I need 'charts', in order to get the top tag and classify each artist I need 'toptagget', cher is my placeholder
toptagsreq=requests.get(f'{url}?method=artist.gettoptags&artist={artist}&api_key={YOUR_API_KEY}&format=json')# gets the top tag for an artist specified in the F STRING {ARTIST} blob, if i change the artist in the line at the beginning it will change from kanye
toptagget=toptagsreq.json() #returns our JSON for the request of the artist tags, I removed the print from this final submission, but it is in assignment 1
chartrequest=requests.get(f'{url}?method=tag.getweeklychartlist&tag=rock&api_key={YOUR_API_KEY}&format=json') 
charts=chartrequest.json()
#toptagget gets changed in the loop that gets the tags, I'm just calling it here so I can reference it at a later point down the line.

#_________________________________________________________________________________________________________________________________________
#BLOCK 2 - Get Artists for Weeks
#This entire loop essentially looks at each week of listening data that we have, and within that week, pulls all the artists up to 250
# and puts them into a master list, for every week. This master list 100% has duplicates but I erase them at a later point (BLOCK 3)
#listylist prints my list, I know you said you didnt want prints so I just turned it into a comment and moved it up here

chartnum=0 # Begins my counter for the loop to see what week we are in
listylist=[] #creates an empty list for the artists to go in later
for i in charts['weeklychartlist']['chart']: #Begins putting us through the large loop of getting weekly data
    fromloop = charts['weeklychartlist']['chart'][chartnum]['from'] #sets beginning of date search
    toloop = charts['weeklychartlist']['chart'][chartnum]['to'] #sets end of date search
    weeklisteningdata = requests.get(f'{url}?method=user.getweeklyartistchart&user=rj&api_key={YOUR_API_KEY}&format=json&to={toloop}&from={fromloop}') #gets my data for the URL that I'm looking at 
    rjdata=weeklisteningdata.json() #this gets the json data that I can parse through and find the lists of artists for every week
    for j in range(0, min(250, len(rjdata['weeklyartistchart']['artist']))): #sets my criteria that for a range between 0 and either 250 or the length of the data (whichever is lower), append the artist to the lsit
        listylist.append(rjdata['weeklyartistchart']['artist'][j]['name']) #
        #print(rjdata['weeklyartistchart']['artist'][j]['name']) #this line isn't necessary, just lets me see the artists I'm dealing with and how many
    chartnum+=1 #advances the chart number counter 1 item, moves onto the next week 
#Breaking here so if you run and something explodes you can stop the kernel at this block.

['Aerosmith',
 'Metallica',
 'Miles Davis',
 "Guns N' Roses",
 'Dream Theater',
 'R.E.M.',
 'Radiohead',
 'Moby',
 'Joe Satriani',
 'Led Zeppelin',
 'Counting Crows',
 'Free',
 'Red Hot Chili Peppers',
 'Jimi Hendrix',
 'Badly Drawn Boy',
 'The Rolling Stones',
 'Portishead',
 'Matchbox Twenty',
 'Jamiroquai',
 'The Beatles',
 'Goo Goo Dolls',
 'Ben Folds Five',
 'Dido',
 'Eels',
 'Stevie Wonder',
 'The Jimi Hendrix Experience',
 'David Bowie',
 'Dire Straits',
 'Rage Against the Machine',
 'A Perfect Circle',
 'Dan Reed Network',
 'Beck',
 'Nine Inch Nails',
 'Ben Harper',
 "Fun Lovin' Criminals",
 'Barenaked Ladies',
 'Jeff Buckley',
 'Norah Jones',
 'Buckcherry',
 'Foo Fighters',
 "King's X",
 'The Doors',
 'The Police',
 'Queen',
 '(hed) Planet Earth',
 'The Cure',
 'U2',
 'Bruce Springsteen',
 'Nirvana',
 'The Smashing Pumpkins',
 'Pink Floyd',
 'Coldplay',
 'Herbie Hancock',
 'The Classic Chill Out Album',
 'Air',
 'Lenny Kravitz',
 'The Velvet Underground',
 'Beth Orton',
 'Juda

In [7]:
#BLOCK 3 - Find + Track Genres
# Urllib.parse was found on stack overflow, I had trouble with Simon & Garfunkel because it contained a character that needed to be
# encoded as "%26", the loop was breaking on that name so i had to encode it so the loop would not break
#https://mojoauth.com/binary-encoding-decoding/percent-encoding-url-encoding-with-python#encoding-query-parameters
#https://www.reddit.com/r/webdev/comments/2c5a6h/im_using_an_api_to_create_qr_codes_i_need_to_put/ 

#Time was a product of Claude, my code kept on stopping due to too many requests, so I asked how I could get around my API's limit and
#using a sleep was the suggestion and it thankfully worked
import urllib.parse as up
import time

#here I make a dictionary so that if an artist is already in the dictionary it skips the loop and passes onto the next artist.
#for the new artists, it looks up the top tag and puts it as the value in the dictionary with the name of the artist as the key
#sickdict #shows me the dictionary at the end of everything, again i disabled it and put it up here so you know its there but doesn't
#run and doesnt look bad.

sickdict={} #makes the empty dictionary for my artists to hide out in 
counter=0 # starts the counter at 0
for item in listylist: #goes through each of the items(artists) in my listylist (Master list of every artist listened to)
    itemencode = up.quote(item) #pencode encodes the name to combat the problem discussed above. (encoding strange characters)
    if item not in sickdict: #checking to see if the artist we found is in the dictionary above, if yes skip to the else condition
        toptagsreq=requests.get(f'{url}?method=artist.gettoptags&artist={itemencode}&api_key={YOUR_API_KEY}&format=json')# gets the top tag for an artist specified in the F STRING {itemencode} blob
        toptagget=toptagsreq.json() #here both toptagget and toptagsreq get reassigned from before (This is what I meant before)
        tags = toptagget['toptags']['tag'] #gets all of the tags for a given artist {itemencode} blob above
        tag = tags[0]['name'] if len(tags) > 0 else 'unknown' # returns the top tag for the artist, returns unknown if no tags exist
        sickdict[item]= tag # puts item in the dictionary as a key, with the tag variable as the value
        #print(item,tag,counter) # Print statement again, just disabling this line so that you know it's there but it doesnt run
    else: #these lines make it so that i'm just moving on in the case that item is already in the dictionary, doesnt waste time
        pass
    counter+=1 #advances the counter by 1 to move onto the next artist.
    time.sleep(0.3) # AI SUGGESTION, this is just so that the API doesn't take too many requests and explode and give me a JSONDecodeError,

#_________________________________________________________________________________________________________________________________________
#BLOCK 4 - Classify Genres
#This loops through each of my genres and puts them into a macrogenre. I encountered a little difficulty with naming the macrogenres
#because some were uppercase and some were lowercase, here i just made them all lowercase to make my life easier and then searched
#for the genre. I know you had a concern that this was maybe AI but it was unfortunately me. The tags made everything really difficult
#becasue I had to rely on people to put the tag as the genre some music people get annoyed when you misnomer their niche band
# and as a result the tags get skewed. If there was a 'Genre' query for the artists in the API I would have used that instead.

for artist in sickdict:
    looptag= sickdict[artist].lower()
    if 'rock' in looptag or 'grunge' in looptag or 'punk' in looptag or 'ska' in looptag or 'alternative' in looptag or 'britpop' in looptag:
        sickdict[artist]='Rock' #very complicated-ly looks for rock,grunge,punk,ska,alternative, or britpop in the looptag, returns Rock if so, if not, moves onto the next genre
    elif 'pop' in looptag:
        sickdict[artist]='Pop' #looks for pop and returns Pop if is in the tag
    elif 'metal' in looptag:
        sickdict[artist]='Metal' #looks for metal and returns Metal if is in the tag
    elif 'jazz' in looptag or 'swing' in looptag or 'blues' in looptag or 'soul' in looptag or 'funk' in looptag:
        sickdict[artist]='Jazz' #looks for jazz,swing,blues,soul,funk and returns Jazz if found. this is another flaw, I have to loop
        #a lot of genres into one big possible miscategory of genre. jazz and funk are pretty different but I have to throw them all into 
        #the category of Jazz because of the listening habits. It wouldnt make sense for me to have an entire funk category because the share is so small
    elif 'indie' in looptag:
        sickdict[artist]='Indie' #looks for indie and returns indie if is in the tag
    elif 'electronic' in looptag or 'techno' in looptag or 'house' in looptag or 'ambient' in looptag or 'trance' in looptag or 'dubstep' in looptag or 'trip-hop' in looptag:
        sickdict[artist]='Electronic' #looks for electronic,techno,house,ambient,trance,dubstep, or trip-hop and returns electronic if any are in the tag
    elif 'classical' in looptag:
        sickdict[artist]='Classical'#looks for indie and returns indie if is in the tag
    elif 'hip hop' in looptag or 'hip' in looptag or 'rap' in looptag or 'hop' in looptag:
        sickdict[artist]='Hip Hop'#looks for hip hop, hip, rap, or hop and returns Hip Hop if any are in the tag
    elif 'folk' in looptag or 'country' in looptag or 'americana' in looptag:
        sickdict[artist] = 'folk' #looks for folk, country, or americana and returns Country if any are in the tag
    else:
        sickdict[artist]='other' #if an artist does not fill out any of these criteria, it will return other, other will get dropped down the line 
#_________________________________________________________________________________________________________________________________________
#BLOCK 5 - Get Data for DataFrame
# This block is the real hard work in my data. It combines everything in the previous blocks into a new pandas dataframe. To overview
# This block gets the artists from each week, references the sickdict to get the name and genre of the artist and tracks genre share and
# total plays on a given week, and then puts it into a list
dataframe=[] #new 
chartnumber=0
url = "http://ws.audioscrobbler.com/2.0/"
YOUR_API_KEY = '915b23d080ae463f6850ac8cec3aefb6'
#Essentially my entire goal is to go through each week, find the total share of each genre during the week,

for chart in charts['weeklychartlist']['chart']:
    fromdate = charts['weeklychartlist']['chart'][chartnumber]['from'] #sets date to begin at for the weeklisteningdata search
    todate = charts['weeklychartlist']['chart'][chartnumber]['to']# sets end date for the weeklisteningdata
    weeklisteningdata = requests.get(f'{url}?method=user.getweeklyartistchart&user=rj&api_key={YOUR_API_KEY}&format=json&to={todate}&from={fromdate}')
   #the above line gets the weekly artist chart for the week specified by the fromdate and todate variables
    weekjson=weeklisteningdata.json() #gets the above request as a JSON print
    weekartists=weekjson['weeklyartistchart']['artist'][:250] #gets first 250 artists, representative of most of the listening for the week
    #essentially tracks each weeks total genreplays and total plays in general
    genreplays={} #creates an empty dictionary for every week in the loop, use this to see genre shares for each of the weeks
    totalplays=0 #initializes the total plays so we can see out of the total plays per week, how many were each genre
    for musician in weekartists: # cycles through each of the top 250 artists per week
        name=musician['name'] #gets artist's name
        plays=int(musician['playcount']) #how many times they were played during this week, gets added to total plays and then calculated as a share of listening 
        tag=sickdict.get(name,'other') #gets the name of the artist from the sick dictionary returns the genre as a tag
        if tag not in genreplays: #if a tag is not in the dictionary, initialize it
            genreplays[tag]=0 #including this to initialize each of the genres within the dict
        genreplays[tag]=genreplays.get(tag) + plays #silly refresher but i forgot how to do this https://www.digitalocean.com/community/tutorials/python-add-to-dictionary
        totalplays+=plays #adds the # of plays to total plays, measures how much the listener is listening per week
        #print(musician['name'],tag, f'week{chartnumber}') # Another print statement so i disabled,prints the musician, their tag, and the chart week number

    # essentially this for loop is putting ALL of my stuff into the empty list so that I can put it into a dataframe
    for genre, plays in genreplays.items(): #gets both genre and playshare from my dictionary (pretty much key and vlaue)
        dataframe.append({'Week':fromdate,'Genre':genre, 'Play Share': plays/totalplays,'Total Plays':totalplays}) 
        #fromdate is in a disgusting format, gonna look this up to change it, https://stackoverflow.com/questions/17594298/date-time-formats-in-python
    chartnumber+=1
    time.sleep(0.3) 

#_________________________________________________________________________________________________________________________________________
#BLOCK 5.5 - Create DataFrame
#actually makes the dataframe and adjusts the date codes and drops the other ugly date column
df = pd.DataFrame(dataframe)
df #calls the function 1
df['Week']=pd.to_datetime(df['Week'], unit='ns') #convert to date, #https://stackoverflow.com/questions/19645706/last-fm-api-date-range
df=df.drop(columns=['date']) #drops date column
df #calls the function 2
df.to_csv('UglyDF.csv') #PUT THIS DATA FRAME IN A CSV SO I CAN STOP REDOING THE ENTIRE THING, goes in the current WD or just the same folder as this doc 

KeyboardInterrupt: 

# Above is all the leg work in order to get the DataFrame Titled 'UglyDF.csv' 
I have uploaded that and the complete DataFrame into the project folder

In [ ]:
#BLOCK 6 - Call the Ugly DataFrame
#skip the long hours of running code, I have a perfectly fine CSV right here.
path='/Users/alex/Documents/Pace/Classes/Junior/Spring/Data Analytics/Project/' #my path, I know it will be different
data=pd.read_csv(f'{path}UglyDF.csv') #brings the data Frame into the notebook from the path
data['Month']=pd.to_datetime(data['Week'], unit='ns').dt.month #changes the Month column so I can sort based on year and month in the future
data['Year']=pd.to_datetime(data['Week'], unit='ns').dt.year #changes the Year column so I can sort based on year and month in the future
data['Play Share (Number)']=data['Play Share'] * data['Total Plays'] #creates a variable so I can see the individual plays for each genre
data #calls the overall data frame into the notebook
#_________________________________________________________________________________________________________________________________________
#BLOCK 7 - Getting Fred Data
#In order to make a Fred Dataframe I need to first get the Data, Thankfully this Package allows me to do just that
# pip install fredapi # this entire thing was shown by this https://pypi.org/project/fredapi/ 
from fredapi import Fred # gets Fred Package
FREDKEY= '2432515d4bc3acaf9cc6bdbbc8ad8558' #gets my fred API key
fred=Fred(api_key=FREDKEY) #structures the variable to use my API key when getting data, this was in the documentation
recession=fred.get_series('USREC') #recession dummy variable
PCEPI=fred.get_series('PCEPI') # PCE variable
UNEMP=fred.get_series('UNRATE') #U3 Unemployment rate
UMICH=fred.get_series('UMCSENT') #UMICH Consumer sentiment Survey
#_________________________________________________________________________________________________________________________________________
#BLOCK 7.5 - Making Fred DataFrame to Merge
#In order to Merge the Fred Data I first need to make another Dataframe with Fred Data in it
freddf=pd.DataFrame(PCEPI)
freddf['REC']=pd.DataFrame(recession) #MAKES A COLUMN FOR EACH VARIABLE
freddf['UNRATE']=pd.DataFrame(UNEMP)
freddf['PCE']=pd.DataFrame(PCEPI)
freddf['UMICH']=pd.DataFrame(UMICH)
freddf.drop(freddf.columns[0],axis=1) #Drops the first initial useless column
freddf['Year']=freddf.index #Makes a column from the year, given by the index but just so i can match it with freds year data
freddf['Month']=pd.to_datetime(freddf['Year'], unit='ns').dt.month #again makes the Month a number so I can sort with it later
freddf['Year']=pd.to_datetime(freddf['Year'], unit='ns').dt.year #again makes the Year a number so I can sort with it later
 #https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html#:~:text=*%20pandas.Series.ffill.%20*%20pandas.Series.fillna.%20*%20pandas.Series.interpolate.
freddf # Calls FRED DataFrame so I can see it 
#_________________________________________________________________________________________________________________________________________
#BLOCK 8 - Merge/Clean Merged DataFrame
#this is to put both of the DataFrames into one that I can export and deal with later for the graphs.
finaldf=pd.merge(data,freddf, on=['Year','Month'], how='left') #merges the two datasets based on Year and Month
finaldf #Merges both of the dataframes into one
finaldf = finaldf[finaldf['Year'] >= 2005] #Excludes FRED data before 2005, the Lastfm api only goes until 2005, but the FRED Data goes to 1959
finaldf.drop(columns=0) #drops an empty column
finaldf = finaldf[finaldf['Year'] < 2026] # Excludes some data to be at 20 years, gets rid of the data where we did not have any values.
finaldf #Calls FinalDF so I can see it
#_________________________________________________________________________________________________________________________________________
#BLOCK 8.5 - Group Data Into Actual Final DataFrame
#This groups everything into my Final Dataframe that I'm going to export and end up using for the Graphs and Regressions. 
groupedfinal=finaldf[['Genre','Play Share','Play Share (Number)','Total Plays','REC', 'UNRATE','PCE','UMICH','Year','Month','Week']].reset_index().groupby(['Week','Genre']).sum()
groupedfinal.to_csv('RealFinalDF.csv')

